# Water Body Segmentation

The First Investigation is Traingig on specific (limited) dataset and evaluation on the other regions as testing areas

### Initializing

In [ ]:
!pip install kornia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 100.2 MB/s eta 0:00:00


In [ ]:
!pip install torchgeo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 871.8/871.8 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 35.2 MB/s eta 0:00:00


In [ ]:
%pip install rasterio

In [ ]:
!pip install -q -U torchgeo

In [ ]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 104.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


In [ ]:
%pip install rasterio

In [ ]:
%pip install utils

  Preparing metadata (setup.py) ... done
  Created wheel for utils: filename=utils-1.0.2-py2.py3-none-any.whl size=13906 sha256=3d01cd2cafb891a39641e1dfbf59d03c9291053bb58e1e1f259071b90f81d4ea
  Stored in directory: /root/.cache/pip/wheels/b6/a1/81/1036477786ae0e17b522f6f5a838f9bc4288d1016fc5d0e1ec
Successfully built utils


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Loading Dataset

In [ ]:
import logging
import os
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchgeo.datasets import stack_samples, RasterDataset
from torchgeo.samplers import RandomGeoSampler, Units
from sklearn.model_selection import train_test_split

class SatelliteDatasetPipeline:
    def __init__(self, root, selected_region, quality, patch_size=512, batch_size=8, crs='EPSG: 3395', res=10):
        self.root = root
        self.selected_region = selected_region
        self.quality = quality
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.crs = crs
        self.res = res

        # Initialize data paths and datasets
        self.data_information = self._load_csv()
        self.high_quality_ids = self._get_high_quality_ids()
        self.s2_path = self._create_paths('S2')
        self.s1_path = self._create_paths('S1')
        self.gt_path = self._create_paths('gt')
        print(f"Dataset size: {len(self.data_information)}")

        print(f"Dataset sizes: S2={len(self.s2_path)}, S1={len(self.s1_path)}, gt={len(self.gt_path)}")

        # Datasets
        self.s2_dataset = self._create_raster_dataset(self.s2_path)
        self.s1_dataset = self._create_raster_dataset(self.s1_path)
        self.gt_dataset = self._create_raster_dataset(self.gt_path, is_image=False)

        # Combined dataset
        self.full_dataset = self.s2_dataset & self.s1_dataset & self.gt_dataset


    def _load_csv(self):
        return pd.read_csv(os.path.join(self.root, 'Revised_information.csv'))

    def _get_high_quality_ids(self):
        region_data = self.data_information[self.data_information['Region Name'] == self.selected_region]
        return region_data[region_data['Quality Check'].isin(self.quality)]['id'].values

    def _create_paths(self, image_type):
        return sorted([os.path.join(self.root, self.selected_region, image_type,
                                    f"{image_type}_{str(i).zfill(5)}.tif") for i in self.high_quality_ids])

    # def _scale_image(self, item):
    #     item['image'] = item['image'] / 10000
    #     return item

    def _scale_image(self, item):
        image = item['image']
        min_val = image.min()
        max_val = image.max()
        item['image'] = ((image - min_val) / (max_val - min_val))
        return item

    def _scale_mask(self, item):
        image = item['mask']
        min_val = image.min()
        max_val = image.max()
        item['mask'] = ((image - min_val) / (max_val - min_val)).to(int)
        return item

    def _create_raster_dataset(self, paths, is_image=True):
        print(paths)
        dataset = RasterDataset(paths=paths,
                             transforms=self._scale_image if is_image else self._scale_mask,
                             crs=self.crs,
                             res=self.res)
        dataset.is_image = is_image
        return dataset


    def train_val_split(self, split_ratio=0.7):
        self.train_size = int(split_ratio * len(self.full_dataset))
        self.val_size = len(self.full_dataset) - self.train_size

    def create_samplers(self):
        self.train_sampler = RandomGeoSampler(self.full_dataset, size=self.patch_size, length=(self.train_size), units=Units.PIXELS)
        self.val_sampler = RandomGeoSampler(self.full_dataset, size=self.patch_size, length=(self.val_size), units=Units.PIXELS)

    def create_dataloaders(self):
        self.train_loader = DataLoader(self.full_dataset, batch_size=self.batch_size, sampler=self.train_sampler, collate_fn=stack_samples)
        self.val_loader = DataLoader(self.full_dataset, batch_size=self.batch_size, sampler=self.val_sampler, collate_fn=stack_samples)

    def get_dataloaders(self):
        return self.train_loader, self.val_loader

# Sequential execution of the pipeline

# Parameters
root = '/content/drive/MyDrive/Patch_Dataset'
quality = ['H', 'L1']
selected_region = 'Southseast Caribbean'
patch_size = 512
batch_size = 8

# Initialize the pipeline
pipeline = SatelliteDatasetPipeline(root=root, selected_region=selected_region, quality=quality, patch_size=patch_size, batch_size=batch_size)

# Split into training and validation sets
pipeline.train_val_split(split_ratio=0.7)

# Create samplers
pipeline.create_samplers()

# Create DataLoaders
pipeline.create_dataloaders()

# Retrieve DataLoaders
train_loader, val_loader = pipeline.get_dataloaders()

batch = next(iter(train_loader))
print(len(train_loader))
batch

In [ ]:
selected_region

In [ ]:
batch = next(iter(train_loader))
batch['image'].shape

### Visualize Dataset

In [ ]:
from typing import Iterable, List
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchgeo.datasets.utils import unbind_samples


def plot_imgs(images: Iterable, axs: Iterable, chnls: List[int] = [3, 2, 1], bright: float = 3.):
    for img, ax in zip(images, axs):
        arr = torch.clamp(bright * img, min=0, max=1).numpy()
        rgb = arr.transpose(1, 2, 0)[:, :, chnls]
        ax.imshow(rgb)
        ax.axis('off')


def plot_msks(masks: Iterable, axs: Iterable):
    for mask, ax in zip(masks, axs):
        ax.imshow(mask.squeeze().numpy(), cmap='Blues')
        ax.axis('off')

def plot_batch(batch: dict, bright: float = 3., cols: int = 4, width: int = 5, chnls: List[int] = [3, 2, 1]):

    # Get the samples and the number of items in the batch
    samples = unbind_samples(batch.copy())

    # if batch contains images and masks, the number of images will be doubled
    n = 2 * len(samples) if ('image' in batch) and ('mask' in batch) else len(samples)

    # calculate the number of rows in the grid
    rows = n//cols + (1 if n%cols != 0 else 0)

    # create a grid
    _, axs = plt.subplots(rows, cols, figsize=(cols*width, rows*width))

    if ('image' in batch) and ('mask' in batch):
        # plot the images on the even axis
        plot_imgs(images=map(lambda x: x['image'], samples), axs=axs.reshape(-1)[::2], chnls=chnls, bright=bright) #type: ignore

        # plot the masks on the odd axis
        plot_msks(masks=map(lambda x: x['mask'], samples), axs=axs.reshape(-1)[1::2]) #type: ignore

    else:

        if 'image' in batch:
            plot_imgs(images=map(lambda x: x['image'], samples), axs=axs.reshape(-1), chnls=chnls, bright=bright) #type: ignore

        elif 'mask' in batch:
            plot_msks(masks=map(lambda x: x['mask'], samples), axs=axs.reshape(-1)) #type: ignore

i = 0
for batch in train_loader:
    # print(batch['image'].shape)
    i += 1
    batch = next(iter(train_loader))
    plot_batch(batch)


### Initialize Model

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
from torch import nn
from sklearn.metrics import accuracy_score, jaccard_score, f1_score
from tqdm.notebook import tqdm
import os
from PIL import Image
from transformers import SegformerForSemanticSegmentation
import pandas as pd
import cv2
import numpy as np
import albumentations as aug

In [ ]:

import os
import tempfile

import kornia.augmentation as K
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from torch import Tensor
from torch.utils.data import DataLoader
from kornia.augmentation import AugmentationSequential
from torchgeo.transforms import indices


# class MinMaxNormalize(K.IntensityAugmentationBase2D):
#     """Normalize channels to the range [0, 1] using min/max values."""

#     def __init__(self, mins: Tensor, maxs: Tensor) -> None:
#         super().__init__(p=1)
#         self.flags = {'mins': mins.view(1, -1, 1, 1), 'maxs': maxs.view(1, -1, 1, 1)}

#     def apply_transform(
#         self,
#         input: Tensor,
#         params: dict[str, Tensor],
#         flags: dict[str, int],
#         transform: Tensor | None = None,
#     ) -> Tensor:
#         return (input - flags['mins']) / (flags['maxs'] - flags['mins'] + 1e-10)
# import torch
# from torch.utils.data import DataLoader

# mean = 0.0
# std = 0.0
# n_samples = 0

# min_val = torch.tensor(float('inf'))  # Large positive value as the initial min
# max_val = torch.tensor(float('-inf'))  # Large negative value as the initial max
# Min = []
# Max = []
# # Iterate through the DataLoader
# for i, batch in enumerate(train_loader):
#     images = batch['image']  # Assuming the images are returned as batch[0]

#     # Reshape to (batch_size, num_channels, height * width) for per-channel statistics
#     images = images.view(images.size(0), images.size(1), -1)

#     for imag in images:
#         Min.append(np.min(np.asanyarray(imag), axis=1))
#         Max.append(np.max(np.asanyarray(imag), axis=1))


#     # Accumulate the sum and squared sum of pixel values for mean and std
#     mean += images.mean([0, 2])  # Mean over batch and pixel dimensions
#     std += images.std([0, 2])    # Std over batch and pixel dimensions

#     n_samples += images.size(0)

# # Final mean, std, min, and max for the whole dataset
# mean /= n_samples
# std /= n_samples

# Min = np.asanyarray(Min)
# Max = np.asanyarray(Max)
# mins = list(np.min(Min, axis=0))
# maxs = list(np.max(Max, axis=0))
# mins = torch.tensor(mins)
# maxs = torch.tensor(maxs)


In [ ]:
transforms = AugmentationSequential(
    # MinMaxNormalize(mins, maxs),
    indices.AppendNDWI(index_green=2, index_nir=8),
    indices.AppendNDVI(index_nir=8, index_red=3),
    K.RandomHorizontalFlip(p=0.5),
    K.RandomVerticalFlip(p=0.5),
    # K.RandomAffine(degrees=(0, 90), p=0.25),
    # K.RandomGaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0), p=0.25),
    # K.RandomResizedCrop(size=(512, 512), scale=(0.8, 1.0), p=0.25),
    data_keys=['image', 'mask'],
)

In [ ]:
batch = next(iter(train_loader))
transformed_batch = transforms(batch)

In [ ]:
# classes = pd.read_csv('classes.csv')['Label']
# id2label = classes.to_dict()
# label2id = {v: k for k, v in id2label.items()}

In [ ]:
from transformers import SegformerForSemanticSegmentation#, SegformerFeatureExtractor
import json
from huggingface_hub import hf_hub_download

# # load id2label mapping from a JSON on the hub
# repo_id = "huggingface/label-files"
# filename = "ade20k-id2label.json"
# id2label = json.load(open(hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset"), "r"))
# id2label = {int(k): v for k, v in id2label.items()}
# label2id = {v: k for k, v in id2label.items()}

# define model
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/mit-b1",
                                                         num_labels=2
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/54.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b1
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.batch_norm.weight                           

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-6)

In [ ]:
batch = next(iter(train_loader))

In [ ]:
import torch
from sklearn.metrics import accuracy_score, jaccard_score, f1_score, precision_score, recall_score
def Accuracy_Assessment(pred_labels, true_labels):
    Accuracy = []
    Accuracy.append(accuracy_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(jaccard_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(f1_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(precision_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(recall_score(pred_labels.flatten(), true_labels.flatten()))

    return Accuracy

In [ ]:
batch = next(iter(train_loader))
torch.where(torch.isnan(batch['image']))

(tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64))

In [ ]:
print(batch['image'].shape)
print(batch['mask'].shape)


torch.Size([8, 15, 512, 512])
torch.Size([8, 1, 512, 512])


In [ ]:
def undesired_mask (transformed_batch):
    # List to store indices of batches with undesired values
    indices_with_undesired = []

    # Iterate over the batches
    for i, batch in enumerate(transformed_batch['mask']):
        # Find unique values in the current batch mask
        unique_values = torch.unique(batch)

        # Check if there are any values other than 0 and 1
        if (unique_values[(unique_values != 0) & (unique_values != 1)]).numel() > 0:
            indices_with_undesired.append(i)
    return indices_with_undesired



In [ ]:
import torch
from tqdm import tqdm
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import torch.nn as nn
from sklearn.metrics import accuracy_score

# Early stopping and learning rate scheduler parameters
patience = 5
min_delta = 1e-5
best_val_loss = float('inf')
best_val_accuracy = 0
trigger_times = 0
best_model_weights = None
Accuracies_train = []
Accuracies_val = []

# Initialize device, optimizer, and scheduler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6, verbose=True)

for epoch in range(1, 50):  # loop over the dataset multiple times
    print(f"Epoch: {epoch}")
    pbar = tqdm(train_loader)
    idb = [13, 14, 15]  # band IDs to select
    train_accuracies = []
    train_losses = []
    val_accuracies = []
    val_losses = []

    model.train()
    Accuracies_t = []
    Accuracies_v = []

    for idx, batch in enumerate(pbar):
        # Augmentation
        transformed_batch = transforms(batch)
        batch_ids = list(np.arange(transformed_batch['image'].shape[0]))

        # Select specified bands
        transformed_batch['image'] = transformed_batch['image'][:, idb, :, :]

        # Identify NaN-containing samples
        batch_nan = torch.unique(torch.where(torch.isnan(transformed_batch['image']))[0]).tolist()
        batch_mask = undesired_mask(transformed_batch)
        batch_nan.extend(batch_mask)
        batch_id = list(set(batch_ids) - set(batch_nan))

        if not batch_id:  # Check if batch_id is empty
            print(f"Warning: Skipping batch {idx} due to all samples containing NaNs or being filtered.")
            continue

        # Remove NaN samples
        pixel_values = transformed_batch['image'][batch_id].to(device)
        labels = transformed_batch["mask"][batch_id].to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(pixel_values=pixel_values, labels=labels)
        upsampled_logits = nn.functional.interpolate(outputs.logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        predicted = upsampled_logits.argmax(dim=1)

        # Calculate accuracy and loss
        pred_labels = predicted.detach().cpu().numpy()
        true_labels = labels.detach().cpu().numpy()
        accuracy = accuracy_score(pred_labels.flatten(), true_labels.flatten())
        loss = outputs.loss

        train_accuracies.append(accuracy)
        train_losses.append(loss.item())
        Accuracies_t.append(Accuracy_Assessment(pred_labels, true_labels))

        pbar.set_postfix({
            'Batch': idx,
            'Pixel-wise accuracy': np.mean(train_accuracies),
            'Loss': np.mean(train_losses)
        })

        # Backward + optimize
        loss.backward()
        optimizer.step()

    Accuracies_train.append(np.mean(np.array(Accuracies_t), axis=0))

    # Validation phase
    model.eval()
    with torch.no_grad():
        for idx, batch in enumerate(val_loader):
            transformed_batch = transforms(batch)
            batch_ids = list(np.arange(transformed_batch['image'].shape[0]))

            # Select specified bands
            transformed_batch['image'] = transformed_batch['image'][:, idb, :, :]

            # Identify NaN-containing samples
            batch_nan = torch.unique(torch.where(torch.isnan(transformed_batch['image']))[0]).tolist()
            batch_mask = undesired_mask(transformed_batch)
            batch_nan.extend(batch_mask)
            batch_id = list(set(batch_ids) - set(batch_nan))

            if not batch_id:
                print(f"Warning: Skipping validation batch {idx} due to all samples containing NaNs or being filtered.")
                continue

            # Remove NaN samples
            pixel_values = transformed_batch['image'][batch_id].to(device)
            labels = transformed_batch["mask"][batch_id].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            upsampled_logits = nn.functional.interpolate(outputs.logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled_logits.argmax(dim=1)

            pred_labels = predicted.detach().cpu().numpy()
            true_labels = labels.detach().cpu().numpy()
            accuracy = accuracy_score(pred_labels.flatten(), true_labels.flatten())
            val_loss = outputs.loss

            val_accuracies.append(accuracy)
            val_losses.append(val_loss.item())
            Accuracies_v.append(Accuracy_Assessment(pred_labels, true_labels))

    Accuracies_val.append(np.mean(np.array(Accuracies_v), axis=0))

    # Calculate average metrics
    avg_train_acc = np.mean(train_accuracies)
    avg_train_loss = np.mean(train_losses)
    avg_val_acc = np.mean(val_accuracies)
    avg_val_loss = np.mean(val_losses)

    print(f"Train Accuracy: {avg_train_acc:.4f} | Train Loss: {avg_train_loss:.4f} | Val Accuracy: {avg_val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")

    # Step the scheduler based on validation loss
    scheduler.step(avg_val_loss)
    print(f'Learning rate after scheduler step: {optimizer.param_groups[0]["lr"]}')

    # Early stopping check with dual criteria
    if avg_val_loss < best_val_loss - min_delta or avg_val_acc > best_val_accuracy + min_delta:
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
        if avg_val_acc > best_val_accuracy:
            best_val_accuracy = avg_val_acc

        best_model_weights = copy.deepcopy(model.state_dict())
        trigger_times = 0  # Reset counter
    else:
        trigger_times += 1
        print(f'Early stopping trigger: {trigger_times}/{patience}')
        if trigger_times >= patience:
            print("Early stopping triggered!")
            break

# Load the best model weights
model.load_state_dict(best_model_weights)

Ac_val = list(np.mean(np.array(Accuracies_val), axis = 0))
Ac_train = list(np.mean(np.array(Accuracies_train), axis = 0))

col = ['Accuracy', 'IOU', 'F1', 'Precision', 'Recall']

path = '/content/drive/MyDrive/Patch_Dataset/Training_Accuracies'
file_name = selected_region + '_1.csv'
file_path = os.path.join(path, file_name)
pd.DataFrame(np.array([Ac_train, Ac_val]), columns = col, index = ['Train', 'Validation']).to_csv(file_path)


# Directory and filename to save the model
model_save_path = f'/content/drive/MyDrive/Patch_Dataset/Models/{selected_region}_1.pth'

# Check if best model weights exist and save them
if best_model_weights is not None:
    torch.save(best_model_weights, model_save_path)
    print(f"Model saved to {model_save_path}")
else:
    print("No model weights to save.")


Epoch: 1


100%|██████████| 8/8 [00:51<00:00,  6.42s/it, Batch=7, Pixel-wise accuracy=0.836, Loss=0.366]


Train Accuracy: 0.8364 | Train Loss: 0.3658 | Val Accuracy: 0.9427 | Val Loss: 0.2148
Learning rate after scheduler step: 0.001
Epoch: 2


100%|██████████| 8/8 [00:23<00:00,  2.88s/it, Batch=7, Pixel-wise accuracy=0.902, Loss=0.29]


Train Accuracy: 0.9017 | Train Loss: 0.2899 | Val Accuracy: 0.8501 | Val Loss: 0.3077
Learning rate after scheduler step: 0.001
Early stopping trigger: 1/5
Epoch: 3


100%|██████████| 8/8 [00:17<00:00,  2.24s/it, Batch=7, Pixel-wise accuracy=0.925, Loss=0.249]


... (29 outputs left out)

100%|██████████| 8/8 [00:18<00:00,  2.35s/it, Batch=7, Pixel-wise accuracy=0.943, Loss=0.183]


Train Accuracy: 0.9430 | Train Loss: 0.1830 | Val Accuracy: 0.9562 | Val Loss: 0.1899
Learning rate after scheduler step: 0.00025
Early stopping trigger: 4/5
Epoch: 19


100%|██████████| 8/8 [00:18<00:00,  2.34s/it, Batch=7, Pixel-wise accuracy=0.963, Loss=0.149]


Train Accuracy: 0.9635 | Train Loss: 0.1489 | Val Accuracy: 0.9587 | Val Loss: 0.1272
Learning rate after scheduler step: 0.00025
Early stopping trigger: 5/5
Early stopping triggered!
Model saved to /content/drive/MyDrive/Patch_Dataset/Models/Southseast Caribbean_1.pth


In [ ]:
Ac_val = list(np.mean(np.array(Accuracies_val), axis = 0))
Ac_train = list(np.mean(np.array(Accuracies_train), axis = 0))

col = ['Accuracy', 'IOU', 'F1', 'Precision', 'Recall']

path = '/content/drive/MyDrive/Patch_Dataset/Training_Accuracies'
file_name = selected_region + '.csv'
file_path = os.path.join(path, file_name)
pd.DataFrame(np.array([Ac_train, Ac_val]), columns = col, index = ['Train', 'Validation']).to_csv(file_path)


# Directory and filename to save the model
model_save_path = f'/content/drive/MyDrive/Patch_Dataset/Models/{selected_region}.pth'

# Check if best model weights exist and save them
if best_model_weights is not None:
    torch.save(best_model_weights, model_save_path)
    print(f"Model saved to {model_save_path}")
else:
    print("No model weights to save.")

Model saved to /content/drive/MyDrive/Patch_Dataset/Models/Persian Gulf.pth


In [ ]:
transformed_batch['image'].shape

torch.Size([8, 16, 512, 512])

### Inference

In [ ]:
import os
import tempfile
import kornia.augmentation as K
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from torch import Tensor
from torch.utils.data import DataLoader

from torchgeo.datasets import EuroSAT100
from torchgeo.transforms import AugmentationSequential, indices

from torch.utils.data import Dataset, DataLoader
# from transformers.optimization import AdamW
import torch
from torch import nn
from sklearn.metrics import accuracy_score, jaccard_score, f1_score
from tqdm.notebook import tqdm
import os
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerFeatureExtractor
import pandas as pd
import cv2
import numpy as np
import albumentations as aug


In [ ]:
import logging
import os
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchgeo.datasets import stack_samples, RasterDataset
from torchgeo.samplers import RandomGeoSampler, Units
from sklearn.model_selection import train_test_split

class SatelliteDatasetPipeline:
    def __init__(self, root, selected_region, quality, patch_size=512, batch_size=8, crs='EPSG: 3395', res=10):
        self.root = root
        self.selected_region = selected_region
        self.quality = quality
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.crs = crs
        self.res = res

        # Initialize data paths and datasets
        self.data_information = self._load_csv()
        self.high_quality_ids = self._get_high_quality_ids()
        self.s2_path = self._create_paths('S2')
        self.s1_path = self._create_paths('S1')
        self.gt_path = self._create_paths('gt')
        # print(f"Dataset size: {len(self.data_information)}")

        # print(f"Dataset sizes: S2={len(self.s2_path)}, S1={len(self.s1_path)}, gt={len(self.gt_path)}")

        # Datasets
        self.s2_dataset = self._create_raster_dataset(self.s2_path)
        self.s1_dataset = self._create_raster_dataset(self.s1_path)
        self.gt_dataset = self._create_raster_dataset(self.gt_path, is_image=False)

        # Combined dataset
        self.full_dataset = self.s2_dataset & self.s1_dataset & self.gt_dataset


    def _load_csv(self):
        return pd.read_csv(os.path.join(self.root, 'Revised_information.csv'))

    def _get_high_quality_ids(self):
        region_data = self.data_information[self.data_information['Region Name'] == self.selected_region]
        return region_data[region_data['Quality Check'].isin(self.quality)]['id'].values

    def _create_paths(self, image_type):
        return sorted([os.path.join(self.root, self.selected_region, image_type,
                                    f"{image_type}_{str(i).zfill(5)}.tif") for i in self.high_quality_ids])

    def _scale_image(self, item):
        image = item['image']
        min_val = image.min()
        max_val = image.max()
        item['image'] = ((image - min_val) / (max_val - min_val))
        return item

    def _scale_mask(self, item):
        image = item['mask']
        min_val = image.min()
        max_val = image.max()
        item['mask'] = ((image - min_val) / (max_val - min_val)).to(int)
        return item

    def _create_raster_dataset(self, paths, is_image=True):
        # print(paths)
        dataset = RasterDataset(paths=paths,
                             transforms=self._scale_image if is_image else self._scale_mask,
                             crs=self.crs,
                             res=self.res)
        dataset.is_image = is_image
        return dataset


    def train_val_split(self, split_ratio=0.7):
        self.test_size = len(self.full_dataset) # min(len(self.full_dataset), 500)
        # self.val_size = 50 # len(self.full_dataset) - self.train_size

    def create_samplers(self):
        self.test_sampler = RandomGeoSampler(self.full_dataset, size=self.patch_size, length=(self.test_size), units=Units.PIXELS)
        # self.val_sampler = RandomGeoSampler(self.full_dataset, size=self.patch_size, length=(self.val_size), units=Units.PIXELS)

    def create_dataloaders(self):
        self.test_loader = DataLoader(self.full_dataset, batch_size=self.batch_size, sampler=self.test_sampler, collate_fn=stack_samples)
        # self.val_loader = DataLoader(self.full_dataset, batch_size=self.batch_size, sampler=self.val_sampler, collate_fn=stack_samples)

    def get_dataloaders(self):
        return self.test_loader

# Sequential execution of the pipeline

# Parameters
# root = '/content/drive/MyDrive/Patch_Dataset'
# quality = ['H']
# selected_region = 'Red Sea'
# patch_size = 512
# batch_size = 8

# # Initialize the pipeline
# pipeline = SatelliteDatasetPipeline(root=root, selected_region=selected_region, quality=quality, patch_size=patch_size, batch_size=batch_size)

# # Split into training and validation sets
# pipeline.train_val_split(split_ratio=0.7)

# # Create samplers
# pipeline.create_samplers()

# # Create DataLoaders
# pipeline.create_dataloaders()

# # Retrieve DataLoaders
# train_loader, val_loader = pipeline.get_dataloaders()

# batch = next(iter(train_loader))
# print(len(train_loader))
# batch

In [ ]:
transforms = AugmentationSequential(
    # MinMaxNormalize(mins, maxs),
    indices.AppendNDWI(index_green=2, index_nir=8),
    indices.AppendNDVI(index_nir=8, index_red=3),
    K.RandomHorizontalFlip(p=0.5),
    K.RandomVerticalFlip(p=0.5),
    # K.RandomAffine(degrees=(0, 90), p=0.25),
    # K.RandomGaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0), p=0.25),
    # K.RandomResizedCrop(size=(512, 512), scale=(0.8, 1.0), p=0.25),
    data_keys=None,
)

<ipython-input-3-49acc13dc63b>:1: DeprecationWarning: Use kornia.augmentation.AugmentationSequential instead
  transforms = AugmentationSequential(


In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerFeatureExtractor
import json
from huggingface_hub import hf_hub_download

# # load id2label mapping from a JSON on the hub
# repo_id = "huggingface/label-files"
# filename = "ade20k-id2label.json"
# id2label = json.load(open(hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset"), "r"))
# id2label = {int(k): v for k, v in id2label.items()}
# label2id = {v: k for k, v in id2label.items()}

# define model
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/mit-b1",
                                                         num_labels=2
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b1 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.lin

In [ ]:
def undesired_mask (transformed_batch):
    # List to store indices of batches with undesired values
    indices_with_undesired = []

    # Iterate over the batches
    for i, batch in enumerate(transformed_batch['mask']):
        # Find unique values in the current batch mask
        unique_values = torch.unique(batch)

        # Check if there are any values other than 0 and 1
        if (unique_values[(unique_values != 0) & (unique_values != 1)]).numel() > 0:
            indices_with_undesired.append(i)
    return indices_with_undesired

In [ ]:
import torch
from sklearn.metrics import accuracy_score, jaccard_score, f1_score, precision_score, recall_score
def Accuracy_Assessment(pred_labels, true_labels):
    Accuracy = []
    Accuracy.append(accuracy_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(jaccard_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(f1_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(precision_score(pred_labels.flatten(), true_labels.flatten()))
    Accuracy.append(recall_score(pred_labels.flatten(), true_labels.flatten()))

    return Accuracy


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def run_inference(model, data_loader, device):
    """
    Run inference on a data loader and return predicted masks and pixel-wise accuracy.

    Args:
        model (torch.nn.Module): The trained segmentation model.
        data_loader (torch.utils.data.DataLoader): DataLoader for inference.
        idb (list): Indices of channels to select from the input image.
        device (torch.device): The device (CPU or GPU) to perform inference on.

    Returns:
        np.array: Array of predicted masks.
        float: Pixel-wise accuracy of the predictions.
    """
    model.eval()  # Set model to evaluation mode
    all_predicted_masks = []
    all_true_masks = []
    accuracies = []
    pbar = tqdm(data_loader)
    rgbs = []
    with torch.no_grad():
        for idx, batch in enumerate(pbar):


            transformed_batch = transforms(batch)
            # print('-----------', batch['image'].shape)

            batch_ids = list(np.arange(transformed_batch['image'].shape[0]))
            n_bands = transformed_batch['image'].shape[1]
            idb = [n_bands-3, n_bands-2, n_bands-1]
            rgb_id = [3, 2, 1]

            # Select specified bands
            image = transformed_batch['image']

            transformed_batch['image'] = transformed_batch['image'][:, idb, :, :]

            # Identify NaN-containing samples
            batch_nan = torch.unique(torch.where(torch.isnan(transformed_batch['image']))[0]).tolist()
            batch_mask = undesired_mask(transformed_batch)
            batch_nan.extend(batch_mask)
            batch_id = list(set(batch_ids) - set(batch_nan))

            if not batch_id:
                # print(f"Warning: Skipping validation batch {idx} due to all samples containing NaNs or being filtered.")
                continue

            # Remove NaN samples
            pixel_values = transformed_batch['image'][batch_id].to(device)
            true_masks = transformed_batch["mask"][batch_id].to(device)

            image = image[batch_id]

            # Forward pass
            outputs = model(pixel_values=pixel_values)

            # Resize logits to match mask size
            upsampled_logits = F.interpolate(outputs.logits, size=true_masks.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled_logits.argmax(dim=1)  # Get the predicted class for each pixel

            # Compute accuracy for this batch
            pred_labels = predicted.detach().cpu().numpy()
            true_labels = true_masks.detach().cpu().numpy()
            accuracy = Accuracy_Assessment(pred_labels, true_labels)

            accuracies.append(accuracy)

            # Store predictions and true masks for inspection
            rgbs.append(image[:, rgb_id, :, :])

            all_predicted_masks.append(pred_labels)
            all_true_masks.append(true_labels)

    avg_accuracy = np.mean(accuracies, axis=0)
    print(f"Average Pixel-wise Accuracy on Inference: Accuracy: {avg_accuracy[0]} | IOU: {avg_accuracy[1]} | F-Score: {avg_accuracy[2]} | Precision: {avg_accuracy[3]} | Recall: {avg_accuracy[4]}")

    # Return all predictions and average accuracy
    return avg_accuracy, all_predicted_masks, all_true_masks, rgbs#, np.array(all_predicted_masks), avg_accuracy
# Usage
# inference_loader = val_loader  # Use validation loader or any other loader for inference

# predicted_masks, avg_accuracy = run_inference(model, inference_loader, idb=idb, device=device)


In [ ]:
from tqdm import tqdm
train_region = 'Ontario Lake'
model_save_path = f'/content/drive/MyDrive/Patch_Dataset/Models/{train_region}.pth'
# Load model weights
model.load_state_dict(torch.load(model_save_path))
# Initialize device, optimizer, and scheduler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)  # Send model to appropriate device

acc = []
IOU = []
f_score = []
precision = []
recall = []
TrainRegion = []
TestRegion = []

patch_size = 512
batch_size = 8

# Parameters
new_root = '/content/drive/MyDrive/Patch_Dataset'
quality = ['H']

selected_region = "Baltic Sea"

# for selected_region in test_regions:
print(f"--------------- Processing region: {selected_region} ---------------")
# Initialize the pipeline
pipeline = SatelliteDatasetPipeline(root=new_root, selected_region=selected_region, quality=quality, patch_size=patch_size, batch_size=batch_size)

# Split into training and validation sets
pipeline.train_val_split(split_ratio=1)

# Create samplers
pipeline.create_samplers()

# Create DataLoaders
pipeline.create_dataloaders()

# Retrieve DataLoaders
inference_loader = pipeline.get_dataloaders()

avg_accuracy, predicted, masks, RGBs = run_inference(model, inference_loader, device=device)
acc.append(avg_accuracy[0])
IOU.append(avg_accuracy[1])
f_score.append(avg_accuracy[2])
precision.append(avg_accuracy[3])
recall.append(avg_accuracy[4])
TrainRegion.append(train_region)
TestRegion.append(selected_region)

# Dict = {'Train Region': TrainRegion, 'Test Region': TestRegion, 'Accuracy': acc, 'IOU': IOU, 'F-Score': f_score, 'Precision': precision, 'Recall': recall}
# df = pd.DataFrame(Dict)
# df.to_csv(f'/content/drive/MyDrive/Patch_Dataset/Test_Accuracies/{train_region}.csv', index=False)



--------------- Processing region: Baltic Sea ---------------


100%|██████████| 66/66 [04:01<00:00,  3.66s/it]

Average Pixel-wise Accuracy on Inference: Accuracy: 0.9018299503931924 | IOU: 0.8431795675223693 | F-Score: 0.9130972780950521 | Precision: 0.9100573582791136 | Recall: 0.9199599172688161


In [ ]:
import matplotlib.pyplot as plt

for idx, (preds, gts, rgbs) in enumerate(zip(predicted, masks, RGBs)):
    for i, (pred, gt, rgb) in enumerate(zip(preds, gts, rgbs)):
      plt.figure(figsize=(10, 3))
      plt.title(f"Batch id: {idx}, image id:{i}")
      plt.subplot(1, 3, 1), plt.imshow(np.transpose(rgb.numpy(), (1, 2, 0))*1.5), plt.xticks([]), plt.yticks([])
      plt.subplot(1, 3, 2), plt.imshow(gt[0], cmap='Blues'), plt.xticks([]), plt.yticks([])
      plt.subplot(1, 3, 3), plt.imshow(pred, cmap='Blues'), plt.xticks([]), plt.yticks([])
      plt.tight_layout()
      # plt.savefig(f"/content/Results_LAS_Pacific/sample{i}.png", dpi=600, bbox_inches="tight")

      plt.show()
    if idx == 30:
      break



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm

b_id = 7
Im_ig = 5
pred, gt, rgb = predicted[b_id][Im_ig], masks[b_id][Im_ig], RGBs[b_id][Im_ig]
# Ensure the output directory exists
save_dir = "/content/"
os.makedirs(save_dir, exist_ok=True)
plt.figure(figsize=(30, 10))

plt.subplot(1, 3, 1)
plt.imshow(np.transpose(rgb.numpy(), (1, 2, 0))*2.5)
plt.xticks([]), plt.yticks([])
# plt.title("RGB Image")

plt.subplot(1, 3, 2)
plt.imshow(gt[0], cmap="Blues")
plt.xticks([]), plt.yticks([])
# plt.title("Ground Truth")

plt.subplot(1, 3, 3)
plt.imshow(pred, cmap="Blues")
plt.xticks([]), plt.yticks([])
# plt.title("Prediction")
# plt.show()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, f"case5_{selected_region}_{train_region}_{b_id}_{Im_ig}.svg"), dpi=600, bbox_inches="tight", format="svg")

plt.show()  # Close to free memory instead of plt.show()

In [ ]:
root = '/content/drive/MyDrive/Patch_Dataset/Models'

test_regions = [region.split('.')[0].split('/')[-1] for region in os.listdir(root)]


In [ ]:
from tqdm import tqdm
train_region = 'Gulf Of Mexico (OSM)'
model_save_path = f'/content/drive/MyDrive/Patch_Dataset/Models/{train_region}.pth'
# Load model weights
model.load_state_dict(torch.load(model_save_path))
# Initialize device, optimizer, and scheduler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)  # Send model to appropriate device

acc = []
IOU = []
f_score = []
precision = []
recall = []
TrainRegion = []
TestRegion = []

patch_size = 512
batch_size = 8

# Parameters
new_root = '/content/drive/MyDrive/Patch_Dataset'
quality = ['H']

for selected_region in test_regions:
    print(f"--------------- Processing region: {selected_region} ---------------")
    # Initialize the pipeline
    pipeline = SatelliteDatasetPipeline(root=new_root, selected_region=selected_region, quality=quality, patch_size=patch_size, batch_size=batch_size)

    # Split into training and validation sets
    pipeline.train_val_split(split_ratio=1)

    # Create samplers
    pipeline.create_samplers()

    # Create DataLoaders
    pipeline.create_dataloaders()

    # Retrieve DataLoaders
    inference_loader = pipeline.get_dataloaders()

    avg_accuracy, predicted, masks = run_inference(model, inference_loader, device=device)
    acc.append(avg_accuracy[0])
    IOU.append(avg_accuracy[1])
    f_score.append(avg_accuracy[2])
    precision.append(avg_accuracy[3])
    recall.append(avg_accuracy[4])
    TrainRegion.append(train_region)
    TestRegion.append(selected_region)

# Dict = {'Train Region': TrainRegion, 'Test Region': TestRegion, 'Accuracy': acc, 'IOU': IOU, 'F-Score': f_score, 'Precision': precision, 'Recall': recall}
# df = pd.DataFrame(Dict)
# df.to_csv(f'/content/drive/MyDrive/Patch_Dataset/Test_Accuracies/{train_region}.csv', index=False)



<ipython-input-25-f1060cce2542>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_save_path))


--------------- Processing region: Great Lakes ---------------


  0%|          | 0/9 [00:02<?, ?it/s]


KeyboardInterrupt: 